# Adaptive Hybrid RecSys — Training v2 (Anti-Collapse Fix)

**Что нового vs `02_train_fast.ipynb`:**
1. **Entropy regularization** на attention gate: `loss += λ_ent · (-H(α))` где H — Shannon entropy, λ_ent = 0.05 по умолчанию.
2. **Live monitoring** attention весов каждую эпоху — видно сразу, есть ли collapse.
3. **Tighter init** для gate: явно инициализируем gate logits к малым значениям (близко к uniform).
4. Сохраняем под `best_full_v2.pt` (не перезаписывает оригинал).

**Цель:** заставить attention gate использовать все 3 компонента (α_s, α_d, α_c близкие к 1/3 в среднем) вместо схлопывания на static.

**Время на T4:** ~75 минут (как и оригинал).

**Требование:** запущен `01_data_pipeline.ipynb`, данные в Drive.

In [ ]:
# ── 1. Drive + paths + GPU check ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, torch
DRIVE_DIR     = '/content/drive/MyDrive/disser'
PROCESSED_DIR = f'{DRIVE_DIR}/data/processed'
OUTPUT_DIR    = f'{DRIVE_DIR}/outputs'
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cudnn.benchmark = True
else:
    print('⚠️ NO GPU — переключи Runtime → T4 GPU!')

In [ ]:
# ── 2. Hyperparameters ────────────────────────────────────────────────────
SUBSAMPLE_FRAC = 1.0      # full data
EMBED_DIM      = 64
MAX_SEQ_LEN    = 50
BATCH_SIZE     = 2048
LR             = 1e-3
EPOCHS         = 30
PATIENCE       = 5
EVAL_USERS     = 5000
EVAL_NEGATIVES = 99
USE_AMP        = True
TFIDF_ON_GPU   = True

# === ANTI-COLLAPSE PARAMS ===
ENTROPY_LAMBDA = 0.05    # λ для entropy regularization (поощряет diverse weights)
LOAD_BALANCE_LAMBDA = 0.01  # дополнительный balance loss (минимизирует variance of mean weights)

CKPT_PATH = f'{OUTPUT_DIR}/checkpoints/best_full_v2.pt'
RESULTS_PATH = f'{OUTPUT_DIR}/results_full_v2.json'
print(f'Subsample: {SUBSAMPLE_FRAC} | epochs: {EPOCHS}')
print(f'λ_entropy: {ENTROPY_LAMBDA} | λ_balance: {LOAD_BALANCE_LAMBDA}')

In [ ]:
# ── 3. Load data + GPU pre-compute ────────────────────────────────────────
import numpy as np, pandas as pd, scipy.sparse as sp, json, gc
from pathlib import Path

P     = Path(PROCESSED_DIR)
stats = json.load(open(P / 'dataset_stats.json'))

train_df       = pd.read_parquet(P / 'train.parquet')
val_df         = pd.read_parquet(P / 'val.parquet')
test_df        = pd.read_parquet(P / 'test.parquet')
train_temporal = pd.read_parquet(P / 'train_temporal.parquet').values.astype(np.float32)
val_temporal   = pd.read_parquet(P / 'val_temporal.parquet').values.astype(np.float32)
test_temporal  = pd.read_parquet(P / 'test_temporal.parquet').values.astype(np.float32)

tfidf_sparse  = sp.load_npz(str(P / 'item_content_sparse.npz'))
seq_data      = np.load(str(P / 'user_sequences.npz'), allow_pickle=True)
user_seqs     = dict(seq_data['sequences'].item())

N_USERS     = stats['n_users']
N_ITEMS     = stats['n_items']
CONTENT_DIM = stats['content_dim']
CONTEXT_DIM = stats['context_dim']

if SUBSAMPLE_FRAC < 1.0:
    n_keep = int(len(train_df) * SUBSAMPLE_FRAC)
    keep_idx = np.random.RandomState(42).choice(len(train_df), n_keep, replace=False)
    train_df = train_df.iloc[keep_idx].reset_index(drop=True)
    train_temporal = train_temporal[keep_idx]

if TFIDF_ON_GPU and DEVICE.type == 'cuda':
    print('Densifying TF-IDF on GPU (fp16)...')
    tfidf_dense = torch.from_numpy(tfidf_sparse.toarray()).half().to(DEVICE)
    pad_row = torch.zeros(1, CONTENT_DIM, dtype=torch.float16, device=DEVICE)
    tfidf_gpu = torch.cat([pad_row, tfidf_dense], dim=0)
    del tfidf_dense
    print(f'TF-IDF on GPU: {tfidf_gpu.shape}, {tfidf_gpu.element_size()*tfidf_gpu.numel()/1e9:.2f} GB')
else:
    tfidf_gpu = None

seq_tensor = np.zeros((N_USERS + 1, MAX_SEQ_LEN), dtype=np.int64)
len_tensor = np.ones(N_USERS + 1, dtype=np.int64)
for uid, seq in user_seqs.items():
    L = min(len(seq), MAX_SEQ_LEN)
    if L > 0:
        seq_tensor[uid, -L:] = seq[-L:]
        len_tensor[uid] = L
seq_tensor_gpu = torch.from_numpy(seq_tensor).to(DEVICE)
len_tensor_gpu = torch.from_numpy(len_tensor).to(DEVICE)

del user_seqs, seq_data, tfidf_sparse
gc.collect()
torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
# ── 4. DataLoader ─────────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class FastRecDataset(Dataset):
    def __init__(self, df, temporal):
        self.users    = df['user_idx'].values.astype(np.int64)
        self.items    = df['item_idx'].values.astype(np.int64)
        self.temporal = temporal
    def __len__(self): return len(self.users)
    def __getitem__(self, idx):
        neg = np.random.randint(1, N_ITEMS + 1)
        return (self.users[idx], self.items[idx], neg,
                self.temporal[idx] if idx < len(self.temporal) else np.zeros(CONTEXT_DIM, np.float32))

def collate_fn(batch):
    users = torch.tensor([b[0] for b in batch], dtype=torch.long)
    pos   = torch.tensor([b[1] for b in batch], dtype=torch.long)
    neg   = torch.tensor([b[2] for b in batch], dtype=torch.long)
    ctx   = torch.tensor(np.stack([b[3] for b in batch]), dtype=torch.float32)
    return users, pos, neg, ctx

common = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True,
              persistent_workers=True, collate_fn=collate_fn)
train_loader = DataLoader(FastRecDataset(train_df, train_temporal), shuffle=True,  drop_last=True, **common)
val_loader   = DataLoader(FastRecDataset(val_df,   val_temporal),   shuffle=False, **common)
test_loader  = DataLoader(FastRecDataset(test_df,  test_temporal),  shuffle=False, **common)

In [ ]:
# ── 5. Model with FIXED gate (small init + return weights for reg) ───────
import torch.nn as nn

class StaticC(nn.Module):
    def __init__(self):
        super().__init__()
        self.user_emb = nn.Embedding(N_USERS + 1, EMBED_DIM, padding_idx=0)
        self.item_emb = nn.Embedding(N_ITEMS + 1, EMBED_DIM, padding_idx=0)
        self.mlp = nn.Sequential(nn.Linear(EMBED_DIM*2, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, EMBED_DIM))
    def forward(self, u, i):
        return self.mlp(torch.cat([self.user_emb(u), self.item_emb(i)], dim=-1))

class DynamicC(nn.Module):
    def __init__(self):
        super().__init__()
        self.item_emb = nn.Embedding(N_ITEMS + 1, EMBED_DIM, padding_idx=0)
        self.gru = nn.GRU(EMBED_DIM, 128, num_layers=2, batch_first=True, dropout=0.1)
        self.proj = nn.Linear(128, EMBED_DIM)
    def forward(self, seqs, lens):
        x = self.item_emb(seqs)
        packed = nn.utils.rnn.pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)
        return self.proj(h[-1])

class ContentC(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(CONTENT_DIM, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, EMBED_DIM))
    def forward(self, x): return self.mlp(x)

class GateFixed(nn.Module):
    """Attention gate с фиксом mode collapse:
       1. Малая init шкала (gate стартует почти uniform)
       2. Возвращает не только веса, но и pre-softmax logits для регуляризации"""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(CONTEXT_DIM, 64)
        self.fc2 = nn.Linear(64, 3)
        # Small init — близко к uniform softmax output
        nn.init.normal_(self.fc2.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.fc2.bias)
    def forward(self, ctx):
        h = torch.relu(self.fc1(ctx))
        logits = self.fc2(h)
        weights = torch.softmax(logits, dim=-1)
        return weights

class HybridFastV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.static  = StaticC()
        self.dynamic = DynamicC()
        self.content = ContentC()
        self.gate    = GateFixed()
        self.head    = nn.Sequential(
            nn.Linear(EMBED_DIM, EMBED_DIM // 2), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(EMBED_DIM // 2, 1))
    def forward_with_gate(self, u, items, seqs, lens, content, ctx):
        h_s = self.static(u, items)
        h_d = self.dynamic(seqs, lens)
        h_c = self.content(content)
        stack = torch.stack([h_s, h_d, h_c], dim=1)  # (B, 3, E)
        w = self.gate(ctx)  # (B, 3)
        fused = (stack * w.unsqueeze(-1)).sum(dim=1)
        score = self.head(fused).squeeze(-1)
        return score, w
    def forward(self, u, items, seqs, lens, content, ctx):
        return self.forward_with_gate(u, items, seqs, lens, content, ctx)[0]

model = HybridFastV2().to(DEVICE)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

# Sanity check — initial gate weights should be near uniform
with torch.no_grad():
    test_ctx = torch.zeros(1, CONTEXT_DIM, device=DEVICE)
    init_w = model.gate(test_ctx).cpu().numpy()[0]
    print(f'Initial gate weights: α_s={init_w[0]:.3f}, α_d={init_w[1]:.3f}, α_c={init_w[2]:.3f}')
    print('(Should be close to 0.333 each — confirms gate starts uniform)')

In [ ]:
# ── 6. Helpers + vectorized eval ──────────────────────────────────────────
def lookup_seqs(uids):    return seq_tensor_gpu[uids], len_tensor_gpu[uids]
def lookup_content(iids): return tfidf_gpu[iids].float()

@torch.no_grad()
def evaluate_fast_with_gate(model, loader, n_users_eval=EVAL_USERS, n_neg=EVAL_NEGATIVES, k=10):
    """Evaluate AND track average gate weights."""
    model.eval()
    rng = np.random.default_rng(42)
    collected = {'u': [], 'pos': [], 'ctx': []}
    n_so_far = 0
    for users, pos, _neg, ctx in loader:
        take = min(len(users), n_users_eval - n_so_far)
        collected['u'].append(users[:take])
        collected['pos'].append(pos[:take])
        collected['ctx'].append(ctx[:take])
        n_so_far += take
        if n_so_far >= n_users_eval: break
    u_all   = torch.cat(collected['u']).to(DEVICE)
    pos_all = torch.cat(collected['pos']).to(DEVICE)
    ctx_all = torch.cat(collected['ctx']).to(DEVICE)
    N = u_all.size(0)

    neg_all = torch.from_numpy(rng.integers(1, N_ITEMS + 1, size=(N, n_neg), dtype=np.int64)).to(DEVICE)
    candidates = torch.cat([pos_all.unsqueeze(1), neg_all], dim=1)
    n_cand = candidates.size(1)
    seqs, lens = lookup_seqs(u_all)

    EVAL_BATCH = 256
    all_scores = torch.zeros(N, n_cand, device=DEVICE)
    all_weights = torch.zeros(N, 3, device=DEVICE)
    for s in range(0, N, EVAL_BATCH):
        e = min(s + EVAL_BATCH, N); b = e - s
        u_e   = u_all[s:e].unsqueeze(1).expand(-1, n_cand).reshape(-1)
        i_e   = candidates[s:e].reshape(-1)
        seq_e = seqs[s:e].unsqueeze(1).expand(-1, n_cand, -1).reshape(b*n_cand, -1)
        len_e = lens[s:e].unsqueeze(1).expand(-1, n_cand).reshape(-1)
        ctx_e = ctx_all[s:e].unsqueeze(1).expand(-1, n_cand, -1).reshape(b*n_cand, -1)
        cnt_e = lookup_content(i_e)
        scores, weights = model.forward_with_gate(u_e, i_e, seq_e, len_e, cnt_e, ctx_e)
        all_scores[s:e] = scores.reshape(b, n_cand)
        all_weights[s:e] = weights.reshape(b, n_cand, 3)[:, 0, :]

    _, ranks = all_scores.sort(dim=1, descending=True)
    pos_rank = (ranks == 0).float().argmax(dim=1)
    hits     = (pos_rank < k).float()
    avg_w    = all_weights.mean(dim=0).cpu().numpy()
    return {
        'Recall@10': hits.mean().item(),
        'NDCG@10': (hits * (1.0 / torch.log2(pos_rank.float() + 2))).mean().item(),
        'alpha_static':  float(avg_w[0]),
        'alpha_dynamic': float(avg_w[1]),
        'alpha_content': float(avg_w[2]),
    }

In [ ]:
# ── 7. Training with anti-collapse regularization ─────────────────────────
import time
from tqdm.notebook import tqdm

def bpr_loss(pos_s, neg_s):
    return -torch.nn.functional.logsigmoid(pos_s - neg_s).mean()

def entropy_regularization(weights, eps=1e-8):
    """Negative entropy: -H(α). High when α uniform (good), low when α concentrated (bad).
    We MAXIMIZE entropy → MINIMIZE negative entropy → so we ADD to loss with positive λ.
    Wait — we want to MINIMIZE total loss, but MAXIMIZE H. So we ADD -H to loss."""
    # H(α) = -sum(α · log α), so -H = sum(α · log α)
    return (weights * torch.log(weights + eps)).sum(dim=-1).mean()

def load_balance_loss(weights):
    """Encourage similar AVERAGE weight across batch for each component.
    Mean weight per component should ideally be ~1/3."""
    mean_per_component = weights.mean(dim=0)  # (3,)
    target = 1.0 / 3.0
    return ((mean_per_component - target) ** 2).sum()

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP and DEVICE.type == 'cuda')

def train_epoch():
    model.train()
    total_bpr, total_ent, total_bal, total_total, n = 0.0, 0.0, 0.0, 0.0, 0
    for users, pos, neg, ctx in tqdm(train_loader, leave=False):
        users = users.to(DEVICE, non_blocking=True)
        pos   = pos.to(DEVICE, non_blocking=True)
        neg   = neg.to(DEVICE, non_blocking=True)
        ctx   = ctx.to(DEVICE, non_blocking=True)
        seqs, lens = lookup_seqs(users)
        pos_cnt = lookup_content(pos)
        neg_cnt = lookup_content(neg)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == 'cuda'):
            B = users.size(0)
            u_cat   = torch.cat([users, users], 0)
            i_cat   = torch.cat([pos, neg], 0)
            seq_cat = torch.cat([seqs, seqs], 0)
            len_cat = torch.cat([lens, lens], 0)
            cnt_cat = torch.cat([pos_cnt, neg_cnt], 0)
            ctx_cat = torch.cat([ctx, ctx], 0)
            scores, weights = model.forward_with_gate(u_cat, i_cat, seq_cat, len_cat, cnt_cat, ctx_cat)
            pos_s, neg_s = scores[:B], scores[B:]
            
            l_bpr = bpr_loss(pos_s, neg_s)
            l_ent = entropy_regularization(weights)   # minimizing this = maximizing entropy
            l_bal = load_balance_loss(weights)
            loss = l_bpr + ENTROPY_LAMBDA * l_ent + LOAD_BALANCE_LAMBDA * l_bal

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_bpr += l_bpr.item(); total_ent += l_ent.item(); total_bal += l_bal.item()
        total_total += loss.item(); n += 1
    return total_total/n, total_bpr/n, total_ent/n, total_bal/n

best_ndcg, patience_cnt, history = 0.0, 0, []
for epoch in range(EPOCHS):
    t0 = time.time()
    tot, bpr, ent, bal = train_epoch()
    scheduler.step()
    val_m = evaluate_fast_with_gate(model, val_loader)
    elapsed = time.time() - t0

    history.append({'epoch': epoch+1, 'loss_total': tot, 'loss_bpr': bpr,
                    'loss_entropy': ent, 'loss_balance': bal, **val_m})
    print(f'Ep {epoch+1:2d}/{EPOCHS} | bpr {bpr:.4f} | -H {ent:+.4f} | bal {bal:.4f} | '
          f'NDCG {val_m["NDCG@10"]:.4f} | α=({val_m["alpha_static"]:.3f}, '
          f'{val_m["alpha_dynamic"]:.3f}, {val_m["alpha_content"]:.3f}) | {elapsed:.0f}s')

    if val_m['NDCG@10'] > best_ndcg:
        best_ndcg = val_m['NDCG@10']; patience_cnt = 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch+1, 'val_metrics': val_m}, CKPT_PATH)
        print(f'  ✓ best saved (NDCG={best_ndcg:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stop @ epoch {epoch+1}')
            break

print(f'\nDone. Best val NDCG@10 = {best_ndcg:.4f}')

In [ ]:
# ── 8. Test eval + save ───────────────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded epoch {ckpt['epoch']}, val NDCG@10={ckpt['val_metrics']['NDCG@10']:.4f}")

test_m = evaluate_fast_with_gate(model, test_loader, n_users_eval=10000, n_neg=99)
print(f'\nTest metrics:')
print(f'  NDCG@10:   {test_m["NDCG@10"]:.4f}')
print(f'  Recall@10: {test_m["Recall@10"]:.4f}')
print(f'  α_static:  {test_m["alpha_static"]:.3f}')
print(f'  α_dynamic: {test_m["alpha_dynamic"]:.3f}')
print(f'  α_content: {test_m["alpha_content"]:.3f}')

# Compare to original (mode-collapsed) model
print('\n=== Comparison vs original (mode-collapsed) model ===')
print(f'{"Model":<25} {"Test NDCG":>10} {"Test Recall":>12} {"α_static":>10}')
print(f'{"original (collapsed)":<25} {0.2807:>10.4f} {0.4384:>12.4f} {0.989:>10.3f}')
print(f'{"v2 (anti-collapse)":<25} {test_m["NDCG@10"]:>10.4f} {test_m["Recall@10"]:>12.4f} {test_m["alpha_static"]:>10.3f}')

results = {
    'model': 'HybridFastV2',
    'entropy_lambda': ENTROPY_LAMBDA,
    'balance_lambda': LOAD_BALANCE_LAMBDA,
    'subsample_frac': SUBSAMPLE_FRAC,
    'test_metrics': test_m,
    'best_val_ndcg': best_ndcg,
    'history': history,
    'n_users': N_USERS, 'n_items': N_ITEMS,
    'total_params': sum(p.numel() for p in model.parameters()),
}
with open(RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved → {RESULTS_PATH}')

In [ ]:
# ── 9. Plot attention weights evolution over epochs ──────────────────────
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
a_s = [h['alpha_static']  for h in history]
a_d = [h['alpha_dynamic'] for h in history]
a_c = [h['alpha_content'] for h in history]
ndcg = [h['NDCG@10'] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, a_s, marker='o', label='α_static',  color='#1565C0', linewidth=2)
axes[0].plot(epochs, a_d, marker='s', label='α_dynamic', color='#FB8C00', linewidth=2)
axes[0].plot(epochs, a_c, marker='^', label='α_content', color='#2E7D32', linewidth=2)
axes[0].axhline(y=1/3, color='gray', linestyle='--', alpha=0.4, label='Uniform (1/3)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Mean attention weight')
axes[0].set_title('Attention weight evolution (anti-collapse v2)', fontsize=12, weight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1)

axes[1].plot(epochs, ndcg, marker='o', color='#C62828', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation NDCG@10')
axes[1].set_title('Validation NDCG@10', fontsize=12, weight='bold')
axes[1].grid(alpha=0.3)
axes[1].axhline(y=0.3221, color='gray', linestyle='--', alpha=0.5, label='Original best (0.3221)')
axes[1].legend()

plt.suptitle(f'Training v2 with entropy regularization (λ_ent={ENTROPY_LAMBDA}, λ_bal={LOAD_BALANCE_LAMBDA})',
             fontsize=12, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figures/v2_training_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Saved → {OUTPUT_DIR}/figures/v2_training_evolution.png')